# Week 25, Day 6: Project review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w025d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('money.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading and showing money and categories."""\nimport re\n\nAMOUNT = re.compile(r"£?\\s*(\\d{1,3}(?:,\\d{3})+|\\d+)(?:\\.(\\d{1,2}))?")\n\n\ndef parse_amount(text):\n    """Text such as \'£1,250.50\' as whole pence. Raises ValueError with a readable message."""\n    text = text.strip()\n    if not text.lstrip("£").strip():\n        raise ValueError("missing amount")\n    m = AMOUNT.fullmatch(text)\n    if m is None:\n        raise ValueError(f"not an amount: {text!r}")\n    total = int(m.group(1).replace(",", "")) * 100 + int((m.group(2) or "").ljust(2, "0"))\n    if total == 0:\n        raise ValueError("amount must be more than £0")\n    return total\n\n\ndef format_pence(pence):\n    """1250 as \'£12.50\', -350 as \'-£3.50\'."""\n    sign = "-" if pence < 0 else ""\n    pounds, rest = divmod(abs(pence), 100)\n    return f"{sign}£{pounds:,}.{rest:02d}"\n\n\ndef clean_category(text):\n    """\' Eating   Out \' as \'eating out\'. Raises ValueError if it isn\'t words of letters."""\n    shown = text.strip()\n    if not shown:\n        raise ValueError("missing category")\n    cleaned = " ".join(shown.lower().split())\n    if len(cleaned) > 20 or not re.fullmatch(r"[a-z]+( [a-z]+)*", cleaned):\n        raise ValueError(f"bad category: {shown!r}")\n    return cleaned\n')
with open('dates.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading dates and months that people type, and stepping through months."""\nimport re\nfrom datetime import datetime, timedelta\n\nFORMATS = ["%Y-%m-%d", "%d/%m/%Y"]\n\n\ndef parse_date(text, today):\n    """\'2026-11-02\', \'02/11/2026\', \'today\', \'yesterday\' or \'\' as ISO text. No future dates."""\n    shown = text.strip()\n    word = shown.lower()\n    if word in ("", "today"):\n        when = today\n    elif word == "yesterday":\n        when = today - timedelta(days=1)\n    else:\n        when = None\n        for fmt in FORMATS:\n            try:\n                when = datetime.strptime(word, fmt).date()\n                break\n            except ValueError:\n                pass\n        if when is None:\n            raise ValueError(f"not a date: {shown!r}")\n    if when > today:\n        raise ValueError(f"date is in the future: {when.isoformat()}")\n    return when.isoformat()\n\n\ndef parse_month(text):\n    """\'2026-11\' or \'11/2026\' as \'2026-11\'. Raises ValueError otherwise."""\n    shown = text.strip()\n    m = re.fullmatch(r"(\\d{4})-(\\d{2})", shown)\n    if m:\n        year, month = int(m.group(1)), int(m.group(2))\n    else:\n        m = re.fullmatch(r"(\\d{1,2})/(\\d{4})", shown)\n        if not m:\n            raise ValueError(f"not a month: {shown!r}")\n        month, year = int(m.group(1)), int(m.group(2))\n    if not 1 <= month <= 12:\n        raise ValueError(f"not a month: {shown!r}")\n    return f"{year}-{month:02d}"\n\n\ndef next_month(month):\n    """\'2026-12\' gives \'2027-01\'."""\n    year, m = int(month[:4]), int(month[5:])\n    if m == 12:\n        return f"{year + 1}-01"\n    return f"{year}-{m + 1:02d}"\n')
with open('core.py', 'w', encoding='utf-8') as f:
    f.write('"""The expense tracker\'s data and the operations on it."""\nfrom money import parse_amount, clean_category\nfrom dates import parse_date, next_month\n\n\ndef new_data():\n    return {"expenses": [], "budgets": {}, "next_id": 1}\n\n\ndef add_expense(data, amount_text, category_text, note, date_text, today):\n    """Check every field, then add the expense, or raise one ValueError listing every problem."""\n    problems = []\n    amount = category = when = None\n    try:\n        amount = parse_amount(amount_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        category = clean_category(category_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        when = parse_date(date_text, today)\n    except ValueError as e:\n        problems.append(str(e))\n    note = note.strip()\n    if len(note) > 60:\n        problems.append("note is longer than 60 characters")\n    if problems:\n        raise ValueError("; ".join(problems))\n    expense = {"id": data["next_id"], "date": when, "amount": amount, "category": category, "note": note}\n    data["expenses"].append(expense)\n    data["next_id"] += 1\n    return expense\n\n\ndef remove_expense(data, expense_id):\n    """Remove and return the expense with this id. KeyError if there isn\'t one."""\n    for i, e in enumerate(data["expenses"]):\n        if e["id"] == expense_id:\n            return data["expenses"].pop(i)\n    raise KeyError(f"no expense #{expense_id}")\n\n\ndef filter_expenses(data, month=None, category=None):\n    """A new list of the matching expenses, newest first (by date, then id)."""\n    chosen = [e for e in data["expenses"]\n              if (month is None or e["date"][:7] == month)\n              and (category is None or e["category"] == category)]\n    return sorted(chosen, key=lambda e: (e["date"], e["id"]), reverse=True)\n\n\ndef total(expenses):\n    return sum(e["amount"] for e in expenses)\n\n\ndef by_category(expenses):\n    """(category, pence) pairs, largest first, alphabetical for ties."""\n    totals = {}\n    for e in expenses:\n        totals[e["category"]] = totals.get(e["category"], 0) + e["amount"]\n    return sorted(totals.items(), key=lambda pair: (-pair[1], pair[0]))\n\n\ndef month_summary(data, month):\n    """Count, total, a row per category with spending or a budget, and the categories over budget."""\n    expenses = filter_expenses(data, month=month)\n    spent = dict(by_category(expenses))\n    rows = []\n    for category in set(spent) | set(data["budgets"]):\n        budget = data["budgets"].get(category)\n        used = spent.get(category, 0)\n        rows.append({"category": category, "spent": used, "budget": budget,\n                     "left": None if budget is None else budget - used})\n    rows.sort(key=lambda r: (-r["spent"], r["category"]))\n    return {"month": month, "count": len(expenses), "total": total(expenses), "rows": rows,\n            "over": [r["category"] for r in rows if r["left"] is not None and r["left"] < 0]}\n\n\ndef monthly_totals(data):\n    """(month, pence) for every month from the first with spending to the last, gaps as 0."""\n    totals = {}\n    for e in data["expenses"]:\n        totals[e["date"][:7]] = totals.get(e["date"][:7], 0) + e["amount"]\n    if not totals:\n        return []\n    out = []\n    month, last = min(totals), max(totals)\n    while month <= last:\n        out.append((month, totals.get(month, 0)))\n        month = next_month(month)\n    return out\n\n\ndef set_budget(data, category_text, amount_text):\n    """Check both, then set the category\'s monthly budget. Returns (category, pence)."""\n    problems = []\n    category = amount = None\n    try:\n        category = clean_category(category_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        amount = parse_amount(amount_text)\n    except ValueError as e:\n        problems.append(str(e))\n    if problems:\n        raise ValueError("; ".join(problems))\n    data["budgets"][category] = amount\n    return category, amount\n\n\ndef remove_budget(data, category_text):\n    """Remove a category\'s budget and return it. KeyError if it has none."""\n    category = clean_category(category_text)\n    if category not in data["budgets"]:\n        raise KeyError(f"no budget for {category!r}")\n    return data["budgets"].pop(category)\n')

### Bug hunt: Review: pounds as floats

This adds up amounts typed as text and returns the total in pence. `add_up(["0.29", "1.15", "4.35"])` should be 579, but it returns 576. Find out why, and fix it.

In [ ]:
def add_up(amount_texts):
    return sum(int(float(t) * 100) for t in amount_texts)

print(add_up(["0.29", "1.15", "4.35"]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"exact totals\"\n    assert add_up([\"0.29\", \"1.15\", \"4.35\"]) == 579\n    assert add_up([\"0.57\", \"2.3\", \"\u00a31,000.07\"]) == 100294\n    assert add_up([str(i / 100) for i in range(1, 200)]) == sum(range(1, 200))\n    assert add_up([]) == 0")

### Bug hunt: Review: a report that rearranges the data

`latest(data, n)` should return the `n` newest expenses without changing anything. It returns the right expenses, but run it and then look at `data["expenses"]`: their order has changed, so the next export or check works on rearranged data. Fix it.

In [ ]:
def latest(data, n):
    data["expenses"].sort(key=lambda e: (e["date"], e["id"]), reverse=True)
    return data["expenses"][:n]

data = {"expenses": [{"id": 1, "date": "2026-10-01", "amount": 5, "category": "a", "note": ""},
                     {"id": 2, "date": "2026-11-01", "amount": 6, "category": "b", "note": ""}],
        "budgets": {}, "next_id": 3}
print(latest(data, 1))
print([e["id"] for e in data["expenses"]])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def D():\n    return {\"expenses\": [{\"id\": i, \"date\": d, \"amount\": 5, \"category\": \"a\", \"note\": \"\"}\n                         for i, d in [(1, \"2026-10-01\"), (2, \"2026-11-01\"), (3, \"2026-10-15\"), (4, \"2026-11-01\")]],\n            \"budgets\": {}, \"next_id\": 5}\ndef test_result():\n    \"the newest n\"\n    assert [e[\"id\"] for e in latest(D(), 3)] == [4, 2, 3] and latest(D(), 0) == [] and len(latest(D(), 10)) == 4\ndef test_unchanged():\n    \"the data isn't changed\"\n    d = D()\n    latest(d, 2)\n    assert d == D()")

### Bug hunt: Review: a forgiving loader

This loader "never crashes": if the file can't be read as JSON, it starts with empty data. Explain to yourself what happens next time the program saves, then fix it so a damaged file raises `ValueError` with a message starting with the path and `is damaged`, while a missing file still gives empty data.

In [ ]:
import json
import os

def load(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return {"expenses": [], "budgets": {}, "next_id": 1}

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_missing():\n    \"a missing file gives empty data\"\n    assert load(\"none.json\") == {\"expenses\": [], \"budgets\": {}, \"next_id\": 1}\ndef test_good():\n    \"a good file loads\"\n    write_file(\"d.json\", '{\"expenses\": [], \"budgets\": {\"food\": 5}, \"next_id\": 7}')\n    assert load(\"d.json\")[\"next_id\"] == 7\ndef test_damaged():\n    \"a damaged file raises, and isn't treated as empty\"\n    write_file(\"d.json\", '{\"expenses\": [{\"id\": 1')\n    try:\n        load(\"d.json\")\n        assert False, \"a damaged file should raise ValueError\"\n    except ValueError as e:\n        assert str(e).startswith(\"d.json is damaged\"), str(e)\ndef test_not_blanket():\n    \"no except Exception\"\n    assert \"except Exception\" not in source and \"except:\" not in source")

### Bug hunt: Review: the hidden clock

`this_month_total(data)` works today, but its tests can only pass in one month of the year, because it asks the computer for the date. Change it to `this_month_total(data, today)`, taking the date as a parameter, with the same behaviour.

In [ ]:
from datetime import date

def this_month_total(data):
    month = date.today().isoformat()[:7]
    return sum(e["amount"] for e in data["expenses"] if e["date"][:7] == month)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from datetime import date as _date\nD = {\"expenses\": [{\"id\": 1, \"date\": \"2026-11-01\", \"amount\": 500, \"category\": \"a\", \"note\": \"\"},\n                  {\"id\": 2, \"date\": \"2026-10-31\", \"amount\": 70, \"category\": \"a\", \"note\": \"\"},\n                  {\"id\": 3, \"date\": \"2025-11-02\", \"amount\": 9, \"category\": \"a\", \"note\": \"\"}], \"budgets\": {}, \"next_id\": 4}\ndef test_any_day():\n    \"works for any today\"\n    assert this_month_total(D, _date(2026, 11, 2)) == 500\n    assert this_month_total(D, _date(2026, 10, 1)) == 70\n    assert this_month_total(D, _date(2027, 1, 1)) == 0\ndef test_no_clock():\n    \"doesn't read the clock\"\n    assert \"today()\" not in source and \"now()\" not in source")

> **What the four have in common.** Each mistake came from a shortcut that looked harmless: `float` because it's the obvious type for "12.50"; sorting in place because it's one line shorter; catching every exception so the program "never crashes"; asking for the date where it's needed. In each case, spend's design already had the safer habit, and the review simply asked: does this code follow it?

### Exercise: Forecast the month

Write `forecast(data, today)` for today's month. For each category with a budget, alphabetically, return a dictionary with `"category"`, `"budget"`, `"spent"` (that month, up to and including today), `"projected"` (`spent * days_in_month // today.day`, whole pence, rounded down), and `"status"`:

- `"over"` if `spent` is already more than the budget;
- `"heading over"` if not, but `projected` is more than the budget;
- `"on track"` otherwise.

Expenses dated after `today` (there shouldn't be any, but data files get edited by hand) don't count.

In [ ]:
import calendar
from datetime import date

def forecast(data, today):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from datetime import date as _date\ndef D():\n    xs = [(1, \"2026-11-01\", 1500, \"food\"), (2, \"2026-11-08\", 1500, \"food\"), (3, \"2026-11-09\", 4000, \"fun\"),\n          (4, \"2026-10-30\", 9999, \"food\"), (5, \"2026-11-10\", 100, \"travel\"), (6, \"2026-11-20\", 5000, \"food\")]\n    return {\"expenses\": [{\"id\": i, \"date\": d, \"amount\": a, \"category\": c, \"note\": \"\"} for i, d, a, c in xs],\n            \"budgets\": {\"food\": 6000, \"fun\": 3000, \"travel\": 5000, \"books\": 1000}, \"next_id\": 7}\ndef test_tenth():\n    \"on 10 November\"\n    got = forecast(D(), _date(2026, 11, 10))\n    assert got == [\n        {\"category\": \"books\", \"budget\": 1000, \"spent\": 0, \"projected\": 0, \"status\": \"on track\"},\n        {\"category\": \"food\", \"budget\": 6000, \"spent\": 3000, \"projected\": 9000, \"status\": \"heading over\"},\n        {\"category\": \"fun\", \"budget\": 3000, \"spent\": 4000, \"projected\": 12000, \"status\": \"over\"},\n        {\"category\": \"travel\", \"budget\": 5000, \"spent\": 100, \"projected\": 300, \"status\": \"on track\"},\n    ], got\ndef test_end_and_february():\n    \"the last day, and a short month\"\n    got = forecast(D(), _date(2026, 11, 30))\n    food = next(r for r in got if r[\"category\"] == \"food\")\n    assert food[\"spent\"] == 8000 and food[\"projected\"] == 8000 and food[\"status\"] == \"over\"\n    d = {\"expenses\": [{\"id\": 1, \"date\": \"2027-02-07\", \"amount\": 700, \"category\": \"fun\", \"note\": \"\"}],\n         \"budgets\": {\"fun\": 2700}, \"next_id\": 2}\n    assert forecast(d, _date(2027, 2, 7))[0][\"projected\"] == 2800\n    assert forecast(d, _date(2027, 2, 7))[0][\"status\"] == \"heading over\"\n    d[\"budgets\"][\"fun\"] = 2800\n    assert forecast(d, _date(2027, 2, 7))[0][\"status\"] == \"on track\"\ndef test_no_budgets():\n    \"no budgets\"\n    assert forecast({\"expenses\": [], \"budgets\": {}, \"next_id\": 1}, _date(2026, 11, 1)) == []")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## What you built

| Module | Lines | Its job | Phase 2 ideas it uses |
|---|---|---|---|
| `money.py` | about 35 | read and show amounts and categories | regular expressions (Week 23), whole pence instead of floats (Week 24) |
| `dates.py` | about 50 | read dates and months; step through months | `datetime` (Weeks 20 and 23), passing in `today` (Week 24) |
| `core.py` | about 120 | add, remove, filter, total, summarise, budgets | decomposition (Week 14), sorting by keys (Week 19), dictionaries and `Counter` (Week 20) |
| `storage.py` | about 110 | load, save atomically, import, export | JSON and CSV (Week 23), files and errors (Phase 1) |
| `spend.py` | about 200 | the command-line interface | `argparse` and exit statuses (Week 21), dispatch with a dictionary |
| Day 5's additions | about 65 | the data file, duplicates, categories, the trend chart | environment variables (Week 21), pairs of loops and O(n²) (Week 17) |
| tests | about 200 | everything above | pytest, fixtures, `tmp_path`, `capsys`, mutation-style checks (Week 24) |

And you used Git to keep its history, and wrote a README (Week 21).

That's a real program. It's also the shape of most programs you'll ever write: input that has to be checked, data that has to be kept safe, rules applied to it, and output a person reads.

## A review checklist

Before calling any project done, go through a list like this one. Answer each honestly for your own `spend`.

**Does it meet the brief?** Go back to Day 1's list of eight requirements and the rules for the data. Try each one. (Did you build all of them? Does `add` really report every problem at once?)

**Is it correct at the edges?** The empty data file, a month with nothing in it, an amount of exactly £0.01, 29 February, a note of exactly 60 characters, a budget spent to the penny.

**Is it safe?** Can any command lose data? What happens if the data file is damaged, or the disk is full during a save, or a statement file doesn't exist?

**Is it tested?** Is every module tested? Would the tests catch the bugs you'd most fear? Do they run in under a second, and pass every time?

**Is it clear?** Does each function do one job, with a name that says what? Does each module have a docstring, and each function that isn't obvious? Could someone new find where to change the rent rule in a minute?

**Is it usable?** Are error messages specific enough to fix the problem without reading the code? Does `--help` explain every command? Does the README get a new person started?

## The decisions that paid off

Several decisions on Day 1 made everything after them easier. They're worth recognising, because you'll make the same choices in other projects:

- **Whole pence.** Not a single rounding bug all week. Compare the first code review below.
- **Check at the edge.** Everything is checked once, as it arrives (typed, or from a file), so the core never has to wonder whether an amount is valid.
- **Pure core, thin shell.** Only `spend.py` prints or reads arguments, so the core and storage were tested with plain function calls, and the interface needed only a handful of end-to-end tests.
- **Pass in the outside world.** `today`, `environ` and `argv` are parameters, so tests could choose any date, environment and command line.
- **All or nothing.** `add_expense` checks everything before changing anything; `save_data` replaces the file only when the new one is complete. A failure never leaves half a change behind.
- **Tests before change.** Day 2's tests made Day 5's refactor a ten-minute job instead of a risk.

## Reviewing code

Reviewing someone else's code (or your own, a week later) is a skill of its own: read it asking "what input would break this?" Each of these functions works on a quick try, and each has a mistake that has caused real bugs in real money software. Fix them.

## One more feature: forecasts

On 10 November, a budget summary says you've spent £30 of a £60 food budget. Sounds fine, but at that rate you'll spend £90 by the end of the month. A **forecast** projects spending to the month's end: spent so far ÷ days so far × days in the month. `calendar.monthrange(year, month)[1]` gives the number of days in a month (Week 20).

It fits the design without changing anything else: a pure function in the core, taking `today`, returning data for the interface to print.

In your own `spend`, wiring it in is the same job as Step 31: a `forecast` subcommand, a `cmd_forecast` that prints each row, and an entry in `COMMANDS`. And a test, of course.

## Where it could go next

Some directions, roughly in the order the course would equip you for them:

- **Recurring expenses**: rent on the 1st of every month, added automatically. A good exercise in dates and in not adding the same thing twice.
- **Objects** (Phase 3): an `Expense` class instead of a dictionary, with `amount` that can only be valid pence, and a `Ledger` that owns the list and the next id. You'll see how much of the checking moves into the classes.
- **A database** (later in the course): SQLite instead of a JSON file, once there are tens of thousands of expenses and you want to ask questions in SQL.
- **A web page or an app**: the core and storage wouldn't change at all. Only a new interface would be written, in place of `spend.py`. That's the strongest argument for the pure-core design.

## Project quiz

## Week 25 review

Tick off each one honestly. If one feels shaky, go back and redo that day's steps.

- I can turn a brief into a plan: the data, the modules and the functions. ([Day 1](w025d1.html))
- I can check input at the edge and collect every problem before rejecting it. ([Day 1](w025d1.html))
- I can filter, group, total and summarise records, and test a module's core with a fixture. ([Day 2](w025d2.html))
- I can load data defensively, save atomically, and import CSV without duplicates. ([Day 3](w025d3.html))
- I can build a command-line interface with argparse, and test it end to end. ([Day 4](w025d4.html))
- I can improve a working program from real use, and refactor it safely under tests. ([Day 5](w025d5.html))
- I can review code for common mistakes, and extend a program within its design. (Today)

Tomorrow is a **rest day**. Next week is the **Phase 2 exam** week: three review days covering recursion, searching and sorting, and tools and testing, then a practice exam, the exam itself, and a debrief.